# 2. NumPy 기초 — 벡터와 행렬을 코드로

> **제2장** · **이론편 대응: 4장 (AI를 위한 수학 I — 선형대수)**
> **예상 소요**: 40분
> **필요 사양**: CPU만으로 충분

---

## 이 장에서 하는 일

이론편 4장에서 손으로 계산했던 것들을 코드로 확인한다. 다루는 값은 다음과 같다.

| 이론편 절 | 손계산 값 | 이 장 |
|---|---|---|
| 4.1 | 28×28 이미지 → 784차원 | 2절 |
| 4.2 | `A @ x = [4, 9]` | 3절 |
| 4.3 | 고유값 λ = 3, 2 | 5절 |
| 4.3 | `A¹⁰v = 3¹⁰v = 59,049v` | 5절 |
| 4.4 | SVD 보존율 99.69% | 6절 |

**중요한 것은 값이 맞는지 확인하는 습관**이다. 코드를 처음 짤 때 어디가 틀렸는지 알기 어려운데,
손으로 구한 값과 대조하면 오류의 위치를 좁힐 수 있다. 이 장부터 그 방식에 익숙해지자.

---

## 1. NumPy를 쓰는 이유

Python에도 리스트가 있는데 왜 NumPy를 따로 쓸까. 답은 **속도**와 **표기**다.

리스트로 두 벡터를 더하려면 반복문이 필요하다.

```python
c = [a[i] + b[i] for i in range(len(a))]
```

NumPy에서는 이렇게 쓴다.

```python
c = a + b
```

표기가 짧아진 것만이 아니다. NumPy는 내부적으로 C로 구현된 연산을 쓰므로 **훨씬 빠르다.**
얼마나 차이 나는지 직접 재 보자.

In [ ]:
import numpy as np
import time

n = 1_000_000

# 파이썬 리스트
a_list = list(range(n))
b_list = list(range(n))

t0 = time.time()
c_list = [a_list[i] + b_list[i] for i in range(n)]
list_time = time.time() - t0

# NumPy 배열
a_np = np.arange(n)
b_np = np.arange(n)

t0 = time.time()
c_np = a_np + b_np
np_time = time.time() - t0

print("=" * 50)
print(f"원소 {n:,}개 덧셈")
print("=" * 50)
print(f"파이썬 리스트 : {list_time*1000:8.1f} ms")
print(f"NumPy 배열   : {np_time*1000:8.1f} ms")
print(f"차이         : {list_time/np_time:8.1f}배")
print()
print(f"결과가 같은가: {c_list[:5] == c_np[:5].tolist()}")

속도 차이가 수십 배에 이른다. 딥러닝은 이런 연산을 수백만 번 반복하므로, 이 차이가 곧
"몇 분"과 "몇 시간"의 차이가 된다.

앞으로 배울 PyTorch도 NumPy와 거의 같은 방식으로 쓰이므로, 여기서 익힌 것이 그대로 이어진다.

---

## 2. 데이터를 벡터로 — 이론편 4.1절

이론편 4.1절에서 "서로 다른 데이터가 벡터라는 하나의 형식으로 바뀐다"고 했다.
28×28 흑백 이미지가 784차원 벡터가 되는 과정을 직접 해 보자.

In [ ]:
import numpy as np

# 28x28 흑백 이미지를 흉내 낸 배열 (실제 이미지는 8장에서 다룬다)
rng = np.random.RandomState(0)
image = rng.randint(0, 256, size=(28, 28))

print("=" * 50)
print("이미지를 벡터로")
print("=" * 50)
print(f"원본 모양   : {image.shape}   ← 28행 28열")
print(f"자료형      : {image.dtype}")
print(f"값의 범위   : {image.min()} ~ {image.max()}")

# 1차원으로 펴기
vector = image.reshape(-1)     # -1은 "알아서 계산하라"는 뜻

print()
print(f"펼친 모양   : {vector.shape}   ← 이론편 4.1절의 784차원")
print(f"28 x 28     = {28*28}")
print(f"앞 10개 값  : {vector[:10]}")

assert vector.shape == (784,), "784차원이 되어야 합니다"
print()
print("[OK] 이론편 4.1절과 일치 — 이미지 한 장이 784개 숫자의 나열이 되었다")

### `reshape(-1)`의 의미

`-1`은 "나머지 차원을 보고 알아서 계산하라"는 뜻이다. 28×28 = 784이므로 `reshape(784)`와 같지만,
크기가 바뀌어도 코드를 고칠 필요가 없어 편리하다.

**여기서 잃는 것도 생각해 보자.** 이론편 12.1절에서 다뤘듯, 이렇게 펼치면 "이 픽셀 옆에 저 픽셀이 있다"는
공간 정보가 사라진다. 이것이 CNN이 필요해진 이유였다. 지금은 일단 벡터로 만드는 것에 집중한다.

---

## 3. 행렬 곱 — 이론편 4.2절 값 검증 ★

이론편 4.2절에서 손으로 계산했던 예제를 그대로 확인한다.

$$A = \begin{pmatrix} 2 & -1 \\ 1 & 3 \end{pmatrix}, \quad \mathbf{x} = \begin{pmatrix} 3 \\ 2 \end{pmatrix}$$

두 가지 방법으로 계산하고 결과가 같은지 본다.

1. 계산 규칙 그대로 (`A @ x`)
2. **열의 선형결합**으로 (`3*a1 + 2*a2`)

In [ ]:
import numpy as np

A = np.array([[2.0, -1.0],
              [1.0,  3.0]])
x = np.array([3.0, 2.0])

print("=" * 50)
print("이론편 4.2절 값 검증")
print("=" * 50)
print(f"A =\n{A}")
print(f"x = {x}")
print()

# 방법 1: 행렬 곱 연산자
y1 = A @ x
print(f"[방법 1] A @ x        = {y1}")

# 방법 2: 열의 선형결합 (이론편 4.2절의 관점)
a1 = A[:, 0]      # 첫째 열
a2 = A[:, 1]      # 둘째 열
y2 = 3 * a1 + 2 * a2
print(f"[방법 2] 3*a1 + 2*a2  = {y2}")
print(f"         a1 = {a1}, a2 = {a2}")
print()

expected = np.array([4.0, 9.0])
print(f"이론편 손계산 값          = {expected}")
print("-" * 50)

assert np.allclose(y1, expected), "방법 1의 결과가 이론편과 다릅니다"
assert np.allclose(y2, expected), "방법 2의 결과가 이론편과 다릅니다"
print("[OK] 두 방법 모두 이론편 4.2절 손계산과 일치")

### `@`와 `*`의 차이 — 흔한 실수

NumPy에서 곱셈 기호가 두 개인데, 뜻이 완전히 다르다.

| 연산자 | 이름 | 하는 일 |
|---|---|---|
| `@` | 행렬 곱 | 선형대수의 행렬 곱셈 |
| `*` | 원소별 곱 | 같은 위치끼리 곱함 |

처음 배울 때 가장 자주 하는 실수가 이 둘을 혼동하는 것이다. 직접 비교해 보자.

In [ ]:
import numpy as np

M = np.array([[1.0, 2.0],
              [3.0, 4.0]])
N = np.array([[5.0, 6.0],
              [7.0, 8.0]])

print("M =\n", M)
print("N =\n", N)
print()
print("M @ N  (행렬 곱)")
print(M @ N)
print("  계산: 1행1열 = 1*5 + 2*7 =", 1*5 + 2*7)
print()
print("M * N  (원소별 곱)")
print(M * N)
print("  계산: 1행1열 = 1*5 =", 1*5)
print()
print("두 결과가 같은가:", np.allclose(M @ N, M * N))
print()
print("→ 신경망의 층 계산은 대부분 @ 를 쓴다 (이론편 4.5절)")

---

## 4. Broadcasting — 모양이 다른 배열끼리의 연산

NumPy의 편리한 기능 중 하나가 **브로드캐스팅**이다. 모양이 다른 배열끼리도
규칙에 맞으면 자동으로 맞춰서 연산해 준다.

예를 들어 (2,3) 행렬에 (3,) 벡터를 더하면, 벡터가 각 행에 자동으로 더해진다.
신경망에서 편향(bias)을 더할 때 이 기능이 쓰인다.

In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])          # (2, 3)
b = np.array([10, 20, 30])         # (3,)

print("=" * 50)
print("Broadcasting")
print("=" * 50)
print(f"a 모양: {a.shape}\n{a}")
print(f"b 모양: {b.shape}\n{b}")
print()
print("a + b =")
print(a + b)
print()
print("→ b가 각 행에 더해졌다. 반복문 없이도 된다.")
print()

# 스칼라도 브로드캐스팅된다
print("a * 2 =")
print(a * 2)
print()

# 규칙에 맞지 않으면 오류
try:
    c = np.array([1, 2])           # (2,) — 열 개수 3과 안 맞음
    _ = a + c
except ValueError as e:
    print("모양이 안 맞으면 오류:")
    print(f"  {e}")

### Broadcasting 규칙

뒤쪽 차원부터 비교해서, 다음 중 하나면 맞출 수 있다.

1. 크기가 같다
2. 둘 중 하나가 1이다

| a 모양 | b 모양 | 결과 | 가능? |
|---|---|---|---|
| (2, 3) | (3,) | (2, 3) | 가능 — 뒤쪽 3이 일치 |
| (2, 3) | (2, 1) | (2, 3) | 가능 — 1은 늘어남 |
| (2, 3) | (2,) | — | **불가** — 3과 2가 안 맞음 |

세 번째가 헷갈리는 지점이다. 각 **행**에 다른 값을 더하고 싶다면 `(2, 1)` 모양으로 만들어야 한다.

In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

# 각 행에 다른 값을 더하고 싶을 때
row_add = np.array([100, 200]).reshape(2, 1)    # (2,) → (2,1)

print("a =\n", a)
print(f"\nrow_add (모양 {row_add.shape}) =\n{row_add}")
print("\na + row_add =")
print(a + row_add)
print("\n→ 1행에 100, 2행에 200이 더해졌다")

---

## 5. 고유값과 고유벡터 — 이론편 4.3절 값 검증 ★

이론편 4.3절에서 특성방정식을 풀어 손으로 구했던 값을 확인한다.

$$A = \begin{pmatrix} 3 & 1 \\ 0 & 2 \end{pmatrix} \;\Rightarrow\; \lambda_1 = 3,\ \lambda_2 = 2$$

그리고 고유벡터가 $\mathbf{v}_1 = (1, 0)$, $\mathbf{v}_2 = (1, -1)$이었다.

In [ ]:
import numpy as np

A = np.array([[3.0, 1.0],
              [0.0, 2.0]])

print("=" * 50)
print("이론편 4.3절 값 검증")
print("=" * 50)
print(f"A =\n{A}\n")

# NumPy로 고유값·고유벡터 구하기
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"계산된 고유값 : {eigenvalues}")
print(f"이론편 손계산    : [3. 2.]")
assert np.allclose(sorted(eigenvalues, reverse=True), [3.0, 2.0])
print("[OK] 고유값 일치\n")

# 고유벡터 확인 — 이론편에서 구한 값으로 직접 검증
print("고유벡터 검증 (정의: Av = λv)")
print("-" * 50)
for lam, v in [(3.0, np.array([1.0, 0.0])),
               (2.0, np.array([1.0, -1.0]))]:
    Av = A @ v
    lam_v = lam * v
    match = np.allclose(Av, lam_v)
    print(f"  λ={lam:.0f}, v={v}")
    print(f"    A @ v = {Av}")
    print(f"    λ * v = {lam_v}")
    print(f"    일치  : {match}")
    assert match
print()
print("[OK] 이론편 4.3절 손계산과 완전히 일치")

### NumPy가 준 고유벡터는 왜 다르게 보이나

위에서 `np.linalg.eig`가 돌려준 고유벡터를 출력해 보면 `[1, 0]`이나 `[1, -1]`이 아니라
`[0.707, -0.707]` 같은 값이 나온다. 틀린 것이 아니다.

**고유벡터는 방향만 정해지고 길이는 정해지지 않는다.** $A\mathbf{v} = \lambda\mathbf{v}$에서
$\mathbf{v}$에 아무 수를 곱해도 등식이 성립하기 때문이다. NumPy는 길이를 1로 맞춰(정규화) 돌려준다.

$(1, -1)$의 길이는 $\sqrt{2} \approx 1.414$이므로, 이것으로 나누면 $(0.707, -0.707)$이 된다.

In [ ]:
import numpy as np

A = np.array([[3.0, 1.0], [0.0, 2.0]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print("NumPy가 돌려준 고유벡터 (열 단위)")
print(eigenvectors.round(4))
print()

for i, lam in enumerate(eigenvalues):
    v = eigenvectors[:, i]
    print(f"λ = {lam:.0f}")
    print(f"  NumPy 벡터     : {v.round(4)}  (길이 {np.linalg.norm(v):.4f})")
    # 첫 성분으로 나누어 이론편 형태로 맞춰 보기
    if abs(v[0]) > 1e-10:
        scaled = v / v[0]
        print(f"  첫 성분으로 나눔: {scaled.round(4)}  ← 이론편의 형태")
    print()

### 고유값의 거듭제곱 — 이론편 4.3절

이론편에서 $A^n\mathbf{v} = \lambda^n\mathbf{v}$를 유도하고, $3^{10} = 59{,}049$를 계산했다.
행렬을 열 번 곱하는 대신 숫자 하나만 거듭제곱하면 된다는 것을 확인해 보자.

이 성질이 왜 중요한지는 이론편 11.2절(그래디언트 소실)과 13.3절(장기 의존성)에서 다뤘다.

In [ ]:
import numpy as np

A = np.array([[3.0, 1.0], [0.0, 2.0]])
v1 = np.array([1.0, 0.0])

print("=" * 50)
print("고유값의 거듭제곱 (이론편 4.3절)")
print("=" * 50)

# 방법 1: 행렬을 10번 곱하기
A10 = np.linalg.matrix_power(A, 10)
result1 = A10 @ v1

# 방법 2: 고유값만 거듭제곱
result2 = (3.0 ** 10) * v1

print(f"[방법 1] A^10 @ v      = {result1}")
print(f"[방법 2] 3^10 * v      = {result2}")
print(f"         3^10          = {3**10:,}")
assert np.allclose(result1, result2)
print("[OK] 두 방법 일치 — 이론편 유도가 맞았다\n")

# 이론편 표: 고유값에 따른 10회·50회 거듭제곱
print("고유값에 따른 변화 (이론편 4.3절 표)")
print("-" * 50)
print(f"{'고유값':<10}{'10회 후':<15}{'50회 후':<15}{'의미'}")
print("-" * 50)
for lam, meaning in [(0.9, "신호 소실"), (1.0, "안정"), (1.1, "신호 폭발")]:
    print(f"{lam:<10}{lam**10:<15.4f}{lam**50:<15.4f}{meaning}")
print("-" * 50)
print("→ 1에서 조금만 벗어나도 반복될수록 차이가 커진다 (이론편 11.2절)")

---

## 6. SVD와 차원 축소 — 이론편 4.4절 값 검증 ★

이론편 4.4절의 학생 성적표 예제를 그대로 확인한다. 학생 4명 × 과목 3개 점수를
**특이값 하나만 남겨** 얼마나 복원되는지 본다.

이론편에서 구한 결과는 다음과 같았다.
- 특이값: 268.05 / 0.84 / 0
- rank-1 근사 최대 오차: 0.47점
- 보존율: 99.69%

In [ ]:
import numpy as np

B = np.array([[90.0, 88.0, 92.0],
              [72.0, 70.0, 74.0],
              [84.0, 82.0, 86.0],
              [60.0, 58.0, 62.0]])

print("=" * 55)
print("이론편 4.4절 값 검증 — 학생 성적표 SVD")
print("=" * 55)
print("원본 B (학생4 x 과목3)")
print(B)
print()

# SVD 분해
U, S, Vt = np.linalg.svd(B, full_matrices=False)

print(f"특이값        : {S.round(3)}")
print(f"이론편 손계산    : [268.051  0.842  0.   ]")
assert np.allclose(S, [268.051, 0.842, 0.0], atol=0.01)
print("[OK] 특이값 일치\n")

# 각 방향이 담은 정보 비율
energy = S**2 / np.sum(S**2) * 100
print(f"정보 비율(%)  : {energy.round(3)}")
print(f"→ 첫 방향 하나가 {energy[0]:.2f}%를 차지\n")

# rank-1 근사: 가장 큰 특이값 하나만 사용
B1 = S[0] * np.outer(U[:, 0], Vt[0])

print("rank-1 근사 결과")
print(B1.round(1))
print()

max_err = np.abs(B - B1).max()
preserve = (1 - np.linalg.norm(B - B1) / np.linalg.norm(B)) * 100

print(f"최대 오차     : {max_err:.3f}점   (이론편: 0.47)")
print(f"보존율        : {preserve:.2f}%   (이론편: 99.69%)")
print("-" * 55)
assert max_err < 0.5, "오차가 이론편보다 큽니다"
assert abs(preserve - 99.69) < 0.1, "보존율이 이론편과 다릅니다"
print("[OK] 이론편 4.4절 손계산과 일치")
print()
print(f"저장량: 원본 {B.size}개 → rank-1 {B.shape[0]+B.shape[1]+1}개")

### 이 결과가 뜻하는 것

12개의 숫자를 8개로 줄였는데 최대 오차가 0.5점도 안 된다. 왜 이런 일이 가능했을까.

원본 표를 보면 **어떤 학생이든 세 과목 점수가 비슷하다.** 학생들 사이의 차이는 주로
"전반적으로 점수가 높은가 낮은가"로 설명된다. SVD는 이 구조를 찾아낸 것이다.

- `U`의 첫 열 → 학생별 전반적 실력
- `Vt`의 첫 행 → 과목별 난이도 패턴

이 둘을 곱하는 것만으로 원본이 거의 복원된다. 이 발상이 이론편 15장 잠재 공간,
20장 Embedding, 22장 LoRA로 이어진다.

In [ ]:
import numpy as np

B = np.array([[90.0, 88.0, 92.0], [72.0, 70.0, 74.0],
              [84.0, 82.0, 86.0], [60.0, 58.0, 62.0]])
U, S, Vt = np.linalg.svd(B, full_matrices=False)

print("SVD가 찾아낸 두 가지 패턴")
print("-" * 55)
print(f"학생별 계수 (U 첫 열 x 특이값):")
student_scores = S[0] * U[:, 0]
for i, sc in enumerate(student_scores, 1):
    print(f"  학생 {i}: {sc:8.1f}")
print()
print(f"과목별 패턴 (Vt 첫 행): {Vt[0].round(4)}")
print()
print("두 값을 곱하면 원본이 복원된다:")
for i in range(4):
    approx = student_scores[i] * Vt[0]
    print(f"  학생 {i+1}: {approx.round(1)}  (실제 {B[i]})")

---

## 7. 정리

### 확인한 이론편 값

| 이론편 절 | 손계산 | 코드 결과 | 일치 |
|---|---|---|---|
| 4.1 | 784차원 | 784 | ✓ |
| 4.2 | `[4, 9]` | `[4. 9.]` | ✓ |
| 4.3 | λ = 3, 2 | 3.0, 2.0 | ✓ |
| 4.3 | 3¹⁰ = 59,049 | 59,049 | ✓ |
| 4.4 | 보존율 99.69% | 99.69% | ✓ |

모든 값이 일치했다. **이론과 코드가 같은 것을 말하고 있다**는 사실을 직접 확인한 셈이다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| `@` vs `*` | 행렬 곱 vs 원소별 곱 — 가장 흔한 실수 |
| `reshape(-1)` | 1차원으로 펴기, 공간 정보는 잃음 |
| Broadcasting | 뒤쪽 차원부터 비교, 크기가 같거나 1이면 가능 |
| 고유벡터 | 방향만 정해짐 — NumPy는 길이 1로 정규화 |
| SVD | 소수의 방향이 정보 대부분을 담을 수 있음 |

### 다음 장

**3. 데이터 시각화 — Matplotlib** — Matplotlib으로 지금까지 다룬 벡터·행렬을 그림으로 본다.
이론편 4.4절의 PCA 그림을 직접 그려 본다.

### 지금까지 확인한 것을 그림으로

숫자만 보면 감이 잘 오지 않는다. **두 가지를 그래프로** 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 반복문 vs 벡터화 ---
ax = axes[0]
sizes = [1000, 10000, 100000, 500000]
loop_times, vec_times = [], []

for n in sizes:
    a = np.random.randn(n)
    b = np.random.randn(n)

    t0 = time.time()
    total = 0.0
    for i in range(n):
        total += a[i] * b[i]
    loop_times.append(time.time() - t0)

    t0 = time.time()
    _ = np.dot(a, b)
    vec_times.append(max(time.time() - t0, 1e-6))

ax.plot(sizes, loop_times, marker="o", linewidth=2,
        color="#DC2626", label="파이썬 반복문")
ax.plot(sizes, vec_times, marker="s", linewidth=2,
        color="#0D9488", label="NumPy 벡터화")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("원소 개수")
ax.set_ylabel("소요 시간 (초)")
ax.set_title("반복문 vs 벡터화")
ax.legend()
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: SVD 에너지 보존 ---
ax = axes[1]
rng = np.random.RandomState(0)
M = rng.randn(50, 30)
U, s, Vt = np.linalg.svd(M, full_matrices=False)
energy = np.cumsum(s ** 2) / np.sum(s ** 2) * 100

ax.plot(range(1, len(energy) + 1), energy, marker="o", markersize=3,
        linewidth=2, color="#1E40AF")
ax.axhline(90, color="gray", linestyle="--", linewidth=1.2)
ax.text(len(energy) * 0.5, 91, "90% 기준", fontsize=8, color="gray")
ax.set_xlabel("사용한 특이값 개수")
ax.set_ylabel("보존된 에너지 (%)")
ax.set_title("특이값 몇 개면 충분한가")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("왼쪽: 원소가 많아질수록 격차가 벌어진다 (양쪽 로그 눈금)")
print(f"  {sizes[-1]:,}개에서 {loop_times[-1]/vec_times[-1]:.0f}배 차이")
print()
n90 = int(np.argmax(energy >= 90)) + 1
print(f"오른쪽: 특이값 {n90}개면 에너지의 90%를 담는다")
print(f"  전체 {len(energy)}개 중 {n90/len(energy)*100:.0f}%만 써도 된다는 뜻")
print("  이론편 4.4절의 저차원 근사가 작동하는 이유다.")